# Locally Linear Embedding
        **unsupervised › dim_reduction** · notebook `lle`

        **Kapan dipakai:** Manifold halus dengan struktur lokal linear.

        **Jebakan umum:** Rapuh pada noise; sering menghasilkan embedding yang terlipat.

**Data demo:** Digits (1797 gambar 8x8, 10 kelas).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/unsupervised/digits.csv"; LABEL_COL = "digit"
NC = 2                         # dimensi hasil (2 untuk visualisasi)
MODEL_KEY = "lle"
OUT = out_dir("unsupervised", "dim_reduction", MODEL_KEY)
SEED = 42; set_seed(SEED)

## 2. Data
Skala [0,1] (MinMax) agar semua metode (termasuk NMF yang butuh non-negatif) berjalan.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.base import clone
from sklearn.manifold import trustworthiness
from superdb.unsupervised import knn_cv_accuracy, plot_embedding
df = pd.read_csv(PATH); y = df[LABEL_COL].to_numpy(); X = MinMaxScaler().fit_transform(df.drop(columns=[LABEL_COL]).to_numpy(float))
print(X.shape, "| kelas:", sorted(set(y)))

## 3. Reduksi

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

reducer = LocallyLinearEmbedding(n_components=NC, n_neighbors=12, random_state=42)
import time; t0 = time.time()
Z = reducer.fit_transform(X, y) if False else reducer.fit_transform(X)
Z = np.asarray(Z); print(f"hasil {Z.shape} dalam {time.time() - t0:.1f} dtk")

## 4. Seberapa baik hasilnya?
- **Trustworthiness** (0-1): apakah tetangga di ruang hasil memang tetangga di ruang asal.
- **Akurasi kNN (CV)** pada representasi ini dibanding data asli: apakah informasi kelas dipertahankan.

In [ ]:
tw = float(trustworthiness(X, Z, n_neighbors=10))
acc_z, acc_raw = knn_cv_accuracy(Z, y), knn_cv_accuracy(X, y)
metrics = {"trustworthiness": tw, f"knn_acc_{Z.shape[1]}D": acc_z, "knn_acc_data_asli": acc_raw}
if hasattr(reducer, "explained_variance_ratio_"): metrics["varians_terjelaskan"] = float(np.sum(reducer.explained_variance_ratio_))
display(show_table(metrics))
plot_embedding(Z, y, f"{MODEL_KEY} ({Z.shape[1]}D)", path=OUT / "embedding.png"); plt.show()

## 5. Berapa dimensi cukup?
Akurasi kNN pada berbagai jumlah dimensi: cari titik di mana akurasi hampir setara data asli.

In [ ]:
rows = []
for nc in [2, 5, 10, 20, 30]:
    if "LinearDiscriminant" in type(reducer).__name__ and nc > len(set(y)) - 1: continue
    try:
        r_ = clone(reducer).set_params(n_components=nc); Zn = np.asarray(r_.fit_transform(X, y) if hasattr(r_, "solver") and "Discriminant" in type(r_).__name__ else r_.fit_transform(X))
        rows.append({"dimensi": nc, "knn_acc": knn_cv_accuracy(Zn, y), "trustworthiness": float(trustworthiness(X, Zn, n_neighbors=10))})
    except Exception as ex:
        print(f"nc={nc} dilewati: {type(ex).__name__}")
sweep = pd.DataFrame(rows).set_index("dimensi"); display(sweep.round(3)); print(f"akurasi kNN data asli (64D): {acc_raw:.3f}")

In [ ]:
pd.DataFrame(Z[:, :min(Z.shape[1], 10)]).assign(label=y).to_csv(OUT / "embedding.csv", index=False)
save_json({"metrics": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- t-SNE/UMAP untuk melihat cluster; PCA/Autoencoder untuk memampatkan sebelum model. Jarak antar cluster di t-SNE tidak bermakna.
- Reduksi yang dipelajari HARUS dilatih hanya pada training bila dipakai sebelum classifier (letakkan di Pipeline).